In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

# Load isotonic regression calibrator (if available)
import os
iso_calibrator_path = '../MODELS/SAVED_MODELS/NGBOOST_PTS_ISOTONIC_CALIBRATOR_PRODUCTION.pkl'
if os.path.exists(iso_calibrator_path):
    isotonic_calibrator = joblib.load(iso_calibrator_path)
    print("✓ Loaded isotonic regression calibrator")
else:
    isotonic_calibrator = None
    print("⚠ Isotonic calibrator not found - using uncalibrated predictions")

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")
print(f"Using isotonic calibration: {isotonic_calibrator is not None}")

✓ Loaded isotonic regression calibrator
Loaded models with calibration factor: 3.97
Using isotonic calibration: True


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_31568/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,John Collins,Over,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
1,PrizePicks,player_points,John Collins,Under,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
2,PrizePicks,player_points,James Harden,Over,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
3,PrizePicks,player_points,James Harden,Under,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
4,PrizePicks,player_points,Franz Wagner,Over,23.5,-137,2025-11-21,2025-11-20T23:41:20Z


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 8 teams with confirmed lineups


### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25, isotonic_calibrator=isotonic_calibrator)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Using isotonic regression calibration
Pre-computing predictions for 58 unique players...


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a cop

Error getting prediction for Paul George: float division by zero


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a cop

,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Justin Edwards,DraftKings,6.5,9.93,Over,100,0,4.94,0.494,High
1,Tristan da Silva,BetRivers,13.5,17.13,Over,114,0,4.93,0.432,High
2,Tristan da Silva,BetRivers,12.5,17.13,Over,-104,1,4.71,0.490,High
3,Tristan da Silva,BetMGM,12.5,17.13,Over,-105,1,4.46,0.468,High
4,Justin Edwards,BetMGM,6.5,9.93,Over,-115,0,4.18,0.480,High


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2,
isotonic_calibrator=isotonic_calibrator)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Using isotonic regression calibration for 2-leg bets
Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 440 valid 2-leg combinations


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])


Applied player frequency limit (2 max appearances per player)
Selected 31 combinations from 440 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Tristan da Silva,Onyeka Okongwu,11.5,13.5,17.13,17.13,over,over,0,4.83,0.241,High,High
1,Tristan da Silva,Luke Kennard,11.5,5.5,17.13,6.97,over,over,0,4.53,0.227,High,Med
2,Onyeka Okongwu,Russell Westbrook,13.5,13.5,17.13,16.61,over,over,0,2.84,0.142,High,High
3,Nickeil Alexander-Walker,Russell Westbrook,16.5,13.5,19.91,16.61,over,over,0,2.27,0.114,High,High
4,Luke Kennard,Bobby Portis,5.5,14.5,6.97,11.54,over,under,0,2.12,0.106,Med,High
5,Nickeil Alexander-Walker,Bobby Portis,16.5,14.5,19.91,11.54,over,under,0,2.05,0.102,High,High
6,Kobe Sanders,VJ Edgecombe,9.5,14.5,11.54,17.42,over,over,0,1.63,0.081,Med,High
7,Kyle Kuzma,VJ Edgecombe,15.5,14.5,17.42,17.42,over,over,0,0.86,0.043,High,High
8,Kobe Sanders,Kyle Kuzma,9.5,15.5,11.54,17.42,over,over,0,0.78,0.039,Med,High
9,Santi Aldama,Trendon Watford,17.5,7.5,15.56,8.32,under,over,0,-0.21,0.000,High,Med


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2,
isotonic_calibrator=isotonic_calibrator)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Using isotonic regression calibration for 2-leg bets
Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players with valid predictions...
Generated 1180 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 51 combinations from 1180 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Tristan da Silva,Justin Edwards,11.5,6.5,17.13,9.93,over,over,0,6.11,0.306,High,High
1,Tristan da Silva,Zaccharie Risacher,11.5,11.5,17.13,15.56,over,over,0,5.68,0.284,High,High
2,Onyeka Okongwu,Justin Edwards,12.5,6.5,17.13,9.93,over,over,0,4.85,0.242,High,High
3,Onyeka Okongwu,Myles Turner,12.5,16.5,17.13,12.97,over,under,0,3.62,0.181,High,High
4,Zaccharie Risacher,Myles Turner,11.5,16.5,15.56,12.97,over,under,0,3.50,0.175,High,High


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2,
isotonic_calibrator=isotonic_calibrator)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Using isotonic regression calibration for 3-leg bets
Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 4666 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 21 combinations from 4666 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Tristan da Silva,Onyeka Okongwu,Nickeil Alexander-Walker,11.5,13.5,16.5,17.13,17.13,19.91,over,over,over,0,8.02,0.160,High,High,High
1,Tristan da Silva,Onyeka Okongwu,Russell Westbrook,11.5,13.5,13.5,17.13,17.13,16.61,over,over,over,0,8.00,0.160,High,High,High
2,Luke Kennard,Nickeil Alexander-Walker,Russell Westbrook,5.5,16.5,13.5,6.97,19.91,16.61,over,over,over,0,4.90,0.098,Med,High,High
3,Luke Kennard,VJ Edgecombe,Bobby Portis,5.5,14.5,14.5,6.97,17.42,11.54,over,over,under,0,4.28,0.086,Med,High,High
4,Kobe Sanders,VJ Edgecombe,Bobby Portis,9.5,14.5,14.5,11.54,17.42,11.54,over,over,under,0,3.80,0.076,Med,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2,
isotonic_calibrator=isotonic_calibrator)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Using isotonic regression calibration for 3-leg bets
Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players with valid predictions...
Generated 20896 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 34 combinations from 20896 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Tristan da Silva,Onyeka Okongwu,Justin Edwards,11.5,12.5,6.5,17.13,17.13,9.93,over,over,over,0,11.42,0.228,High,High,High
1,Tristan da Silva,Zaccharie Risacher,Justin Edwards,11.5,11.5,6.5,17.13,15.56,9.93,over,over,over,0,11.08,0.222,High,High,High
2,Goga Bitadze,Onyeka Okongwu,Zaccharie Risacher,4.5,12.5,11.5,5.74,17.13,15.56,over,over,over,0,8.03,0.161,Low,High,High
3,Goga Bitadze,Nickeil Alexander-Walker,Myles Turner,4.5,16.5,16.5,5.74,19.91,12.97,over,over,under,0,5.07,0.101,Low,High,High
4,Nickeil Alexander-Walker,Luke Kennard,Myles Turner,16.5,5.5,16.5,19.91,6.97,12.97,over,over,under,0,4.97,0.099,High,Med,High
